# Day 1: Data Curation — Vietnamese Price Prediction Dataset

Pipeline: Load 121K products from Tiki JSONL -> Clean (9-step Vietnamese pipeline) -> Dedup -> EDA -> Weighted Sampling -> Split -> Push to HuggingFace Hub

**Dataset:** 54 JSONL files (48 Tiki scraper + 6 Kaggle)
**Target:** ~110K items, split 100K/5K/5K (train/val/test)
**HuggingFace:** SeanSunny/items_raw_tv

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from pricer_vi.items import Item
from pricer_vi.parser import parse

In [ ]:
# Configuration
DATA_DIR = Path("../Tiki/Tiki_dataset_scrape")
RANDOM_SEED = 42
HF_DATASET_NAME = "SeanSunny/items_raw_tv"
TRAIN_SIZE = 100_000
VAL_SIZE = 5_000
TEST_SIZE = 5_000

## Step 1: Load all JSONL files

In [ ]:
items = []
for filepath in sorted(DATA_DIR.glob("*.jsonl")):
    is_kaggle = "kaggle" in filepath.name
    count = 0
    for line in open(filepath, encoding="utf-8"):
        datapoint = json.loads(line)
        if is_kaggle:
            category = "Th\u1eddi Trang"
        else:
            raw_cat = datapoint.get("category", "")
            category = raw_cat.split(" > ")[0] if " > " in raw_cat else raw_cat
        item = parse(datapoint, category)
        if item:
            items.append(item)
            count += 1
    print(f"  {filepath.name}: {count:,} items")

print(f"\nTotal loaded: {len(items):,} items")

In [ ]:
# Inspect a sample item
items[0]

In [ ]:
print(items[0].full[:500])

## Step 2: EDA (before dedup)

In [ ]:
lengths = [len(item.full) for item in items]
prices = [item.price for item in items]

In [ ]:
# Text length distribution
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel("Length (chars)")
plt.ylabel("Count")
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, min(max(lengths)+100, 6000), 100))
plt.show()

In [ ]:
# Price distribution (log scale for VND)
plt.figure(figsize=(15, 6))
plt.title(f"Prices (VND): Avg {sum(prices)/len(prices):,.0f}, Median {int(np.median(prices)):,}\n")
plt.xlabel("Price (VND)")
plt.ylabel("Count")
log_bins = np.logspace(np.log10(max(min(prices), 1)), np.log10(max(prices) + 1), 50)
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=log_bins)
plt.xscale("log")
plt.show()

## Step 3: Deduplication

In [ ]:
random.seed(RANDOM_SEED)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication: {len(items):,} items")

## Step 4: EDA (after dedup)

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel("Length (chars)")
plt.ylabel("Count")
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices (VND): Avg {sum(prices)/len(prices):,.0f}, Median {int(np.median(prices)):,}\n")
plt.xlabel("Price (VND)")
plt.ylabel("Count")
log_bins = np.logspace(np.log10(max(min(prices), 1)), np.log10(max(prices) + 1), 50)
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=log_bins)
plt.xscale("log")
plt.show()

In [ ]:
# Category distribution
category_counts = Counter([item.category for item in items])
categories = list(category_counts.keys())
counts = [category_counts[c] for c in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title("How many in each category")
plt.xlabel("Categories")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()

## Step 5: Weighted Sampling

In [ ]:
np.random.seed(RANDOM_SEED)

target_size = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
sample_size = min(len(items), target_size)

prices_arr = np.array([it.price for it in items], dtype=float)
categories_arr = np.array([it.category for it in items])

# Normalize prices to [0, 1]
p = (prices_arr - prices_arr.min()) / (prices_arr.max() - prices_arr.min() + 1e-9)

# Price^2 weighting
w = p ** 2

# Category penalties (uncomment after reviewing EDA)
# w[categories_arr == "Th\u1eddi Trang"] *= 0.3

# Normalize
w = w / w.sum()

idx = np.random.choice(len(items), size=sample_size, replace=False, p=w)
sample = [items[i] for i in idx]
print(f"Sample size: {len(sample):,}")

In [ ]:
# Price distribution after sampling
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices (VND): Avg {sum(prices)/len(prices):,.0f}, Median {int(np.median(prices)):,}\n")
plt.xlabel("Price (VND)")
plt.ylabel("Count")
log_bins = np.logspace(np.log10(max(min(prices), 1)), np.log10(max(prices) + 1), 50)
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=log_bins)
plt.xscale("log")
plt.show()

In [ ]:
# Final shuffle
random.seed(RANDOM_SEED)
random.shuffle(sample)

In [ ]:
# Category distribution after sampling
category_counts = Counter([item.category for item in sample])
categories = list(category_counts.keys())
counts = [category_counts[c] for c in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title("How many in each category (after sampling)")
plt.xlabel("Categories")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()

In [ ]:
# Category pie chart
plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct="%1.0f%%", startangle=90)
centre_circle = plt.Circle((0, 0), 0.70, fc="white")
plt.gcf().gca().add_artist(centre_circle)
plt.title("Categories")
plt.axis("equal")
plt.show()

In [ ]:
# Price vs text length scatter
sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")
plt.xlabel("Text length (chars)")
plt.ylabel("Price (VND)")
plt.title("Is there a simple correlation with text length?")
plt.show()

## Step 6: Split and Push to HuggingFace Hub

In [ ]:
test = sample[:TEST_SIZE]
val = sample[TEST_SIZE:TEST_SIZE + VAL_SIZE]
train = sample[TEST_SIZE + VAL_SIZE:]
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("Set HF_TOKEN in .env or login manually: login('your_token')")

In [ ]:
Item.push_to_hub(HF_DATASET_NAME, train, val, test)
print(f"Pushed to {HF_DATASET_NAME}")